In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from utils import Silver


required_bronze_tables = ['Green','Yellow']

silver = Silver('GY_PRE_VALIDATION', 'GY_VALID', 'GY_INVALID', required_bronze_tables)

green_df = silver.bronze_tables_dfs['Green']
yellow_df = silver.bronze_tables_dfs['Yellow']

green_df = green_df.select(
    green_df.VendorID.alias('VendorId'),
    green_df.lpep_pickup_datetime.alias('PickUpDateTime'),
    green_df.lpep_dropoff_datetime.alias('DropOffDateTime'),
    green_df.PULocationID.alias('PickUpLocationId'),
    green_df.DOLocationID.alias('DropOffLocationId'),
    green_df.passenger_count.alias('PassengerCount'),
    green_df.trip_distance.alias('TripDistance'),
    green_df.tip_amount.alias('TipAmount'),
    green_df.total_amount.alias('TotalAmount')
)

yellow_df = yellow_df.select(
    yellow_df.VendorID.alias('VendorId'),
    yellow_df.tpep_pickup_datetime.alias('PickUpDateTime'),
    yellow_df.tpep_dropoff_datetime.alias('DropOffDateTime'),
    yellow_df.PULocationID.alias('PickUpLocationId'),
    yellow_df.DOLocationID.alias('DropOffLocationId'),
    yellow_df.passenger_count.alias('PassengerCount'),
    yellow_df.trip_distance.alias('TripDistance'),
    yellow_df.tip_amount.alias('TipAmount'),
    yellow_df.total_amount.alias('TotalAmount')
)

id_window = Window.orderBy(F.monotonically_increasing_id())
GY_Pre_validation_df = green_df.unionAll(yellow_df).withColumn("Id", F.row_number().over(id_window)) # adding unique Id to help testing

1446358

In [2]:
# pyspark version 4.0.0 has try_cast to set not converting values to null instead of causing error (for this dataset this is not the case).
validation_df = GY_Pre_validation_df.select(
    GY_Pre_validation_df.Id,
    GY_Pre_validation_df.VendorId,
    GY_Pre_validation_df.PickUpDateTime,
    GY_Pre_validation_df.DropOffDateTime,
    GY_Pre_validation_df.PickUpLocationId,
    GY_Pre_validation_df.DropOffLocationId,
    GY_Pre_validation_df.PassengerCount.cast('int'),
    GY_Pre_validation_df.TripDistance.cast('float'),
    GY_Pre_validation_df.TipAmount.cast('float'),
    GY_Pre_validation_df.TotalAmount.cast('float'),
)\
.na.fill('999',["VendorId"])

# Intial data analysis showed negative TotalAmounts this check tells us if TotalAmount is causing the dup values
window = Window.partitionBy('VendorId','PickUpDateTime','DropOffDateTime','PickUpLocationId','DropOffLocationId').orderBy(F.col('TotalAmount').desc())
dups_validation_df = validation_df.withColumn('row', F.row_number().over(window))

validation_columns_df = dups_validation_df\
.withColumn('IsValid', 
            F.when(
                (F.col('PassengerCount') == 0) |
                (F.col('PassengerCount').isNull()), False
            ).otherwise(True)
            )\
.withColumn('IsDuplicate', F.when(F.col('row') > 1, True).otherwise(False))

In [3]:
GY_Validated_df = validation_columns_df.filter((F.col('IsValid') == True) & (F.col('IsDuplicate') == False)).drop('row', 'IsValid', 'IsDuplicate')

In [4]:
GY_Invalid_dups_df = validation_columns_df.filter((F.col('IsValid') == False) | (F.col('IsDuplicate') == True)).drop('row')

In [5]:
if __name__ == '__main__':
    silver.create_silver(GY_Pre_validation_df, GY_Validated_df, GY_Invalid_dups_df)

    silver.run_test(return_dfs=False) # if set to True all 3 dataframes are returned for further investigation

    silver.stop_spark()

--------------------------------------------------
Writing into silver: GY_PRE_VALIDATION
Successfully saved data in location:
./PipelineData/Silver/GY_PRE_VALIDATION
--------------------------------------------------
Writing into silver: GY_VALID
Successfully saved data in location:
./PipelineData/Silver/GY_VALID
--------------------------------------------------
Writing into silver: GY_INVALID
Successfully saved data in location:
./PipelineData/Silver/GY_INVALID
--------------------------------------------------
Running test to check if counts match...
Counts match: 1446358
GY_PRE_VALIDATION: 1446358
Union of GY_VALID and GY_INVALID: 1446358


In [6]:
# required_bronze_tables = ['Green','Yellow']
# silver = Silver('GY_PRE_VALIDATION', 'GY_VALID', 'GY_INVALID', required_bronze_tables)
# pre_validation_df, valid_df, invalid_df = silver.run_test(return_dfs=True)

--------------------------------------------------
Running test to check if counts match...
Counts match: 1446358
GY_PRE_VALIDATION: 1446358
Union of GY_VALID and GY_INVALID: 1446358


In [7]:
# pre_validation_df.filter(F.col('Id').isin('1', '50')).show(2)

# valid_df.filter(F.col('Id') == '1').show(1)
# invalid_df.filter(F.col('Id') == '50').show(1)

+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|VendorId|     PickUpDateTime|    DropOffDateTime|PickUpLocationId|DropOffLocationId|PassengerCount|TripDistance|TipAmount|TotalAmount| Id|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|       2|2021-01-01 00:15:56|2021-01-01 00:19:52|              43|              151|             1|        1.01|        0|        6.8|  1|
|       1|2021-01-01 01:01:59|2021-01-01 01:10:50|              14|               14|             0|         .00|        0|        7.8| 50|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+

+---+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+
| Id|VendorId|     